In [1]:
import torch
import torch.nn as nn
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from statsmodels.tsa.stattools import grangercausalitytests
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops
from scipy.sparse import coo_matrix
import time
import math
import random
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm

# --- Constants ---
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_with_objects_reduced.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
TRAIN_PCT, VAL_PCT = 0.8, 0.1
BATCH_SIZE = 32

# --- Tokenizer and PAD_ID ---
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size
NUM_COLORS = 77
NUM_CATEGORIES = 53
NUM_OBJECTS = 61

# --- Granger Causality Matrix Creation ---
def create_granger_causality_matrix(eeg_batch):
    eeg_sample = eeg_batch[0].cpu().numpy().T
    num_channels = eeg_sample.shape[1]
    causality_matrix = np.zeros((num_channels, num_channels))

    for i in range(num_channels):
        for j in range(num_channels):
            if i == j:
                continue
            ts_i = eeg_sample[:, i]
            ts_j = eeg_sample[:, j]
            data = np.vstack([ts_j, ts_i]).T
            try:
                results = grangercausalitytests(data, maxlag=5, verbose=False)
                p_value = results[5][0]['ssr_ftest'][1]
                if p_value < 0.05:
                    causality_matrix[i, j] = 1.0
            except:
                causality_matrix[i, j] = 0.0

    adj_matrix = coo_matrix(causality_matrix)
    edge_index, edge_attr = from_scipy_sparse_matrix(adj_matrix)
    
    return edge_index.to(torch.long), edge_attr.to(torch.float)

# --- Dataset and DataLoader ---
class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
        
        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype('float32'))
        # The metadata shape is now 4
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype('float32'))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype('int64'))
        
        return eeg, meta, text

def collate_multimodal_batch(batch):
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, txt in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(txt)

    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=PAD_ID)

    return eeg_batch.float(), meta_batch.float(), text_padded

# Create Dataset and Loaders
dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
N = len(dataset)
n_train = int(N * TRAIN_PCT)
n_val   = int(N * VAL_PCT)
n_test  = N - n_train - n_val
g = torch.Generator().manual_seed(42)
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=g)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_multimodal_batch)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)

# Execute the matrix creation process
eeg_b, _, _ = next(iter(train_loader))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Creating a static Granger Causality matrix on {device}...")
granger_edge_index, granger_edge_attr = create_granger_causality_matrix(eeg_b)
num_channels = eeg_b.shape[1]

# Add self-loops to handle empty graphs
if granger_edge_index.numel() == 0:
    print("Warning: Generated Granger matrix is empty. Creating a fallback graph with self-loops.")
    granger_edge_index = torch.arange(num_channels, dtype=torch.long).unsqueeze(0).repeat(2, 1)
    granger_edge_attr = torch.ones(num_channels, dtype=torch.float)

granger_edge_index, granger_edge_attr = add_self_loops(granger_edge_index, edge_attr=granger_edge_attr, num_nodes=num_channels)

# Correct the data types before moving to the device
granger_edge_index = granger_edge_index.to(torch.long)
granger_edge_attr = granger_edge_attr.to(torch.float32)

granger_edge_index = granger_edge_index.to(device)
granger_edge_attr = granger_edge_attr.to(device)

print("Granger Matrix created and loaded to device.")

/home/poorna/venvs/torch/lib64/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


Creating a static Granger Causality matrix on cuda...


/home/poorna/venvs/torch/lib64/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


Granger Matrix created and loaded to device.


In [2]:
# --- Component 1: SpatioTemporalEEGEncoder (No changes) ---
class SpatioTemporalEEGEncoder(nn.Module):
    # ... (no changes here, same as your code)
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = GCNConv(num_channels, enc_hidden)
        self.gcn2 = GCNConv(enc_hidden, enc_hidden)
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers, bidirectional=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg, edge_index, edge_attr):
        batch_size = eeg.shape[0]
        num_timesteps = eeg.shape[2]
        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size)
        batch_offset = torch.arange(batch_size, device=eeg.device) * self.num_channels
        batch_edge_index += batch_offset.repeat_interleave(edge_index.shape[1]).unsqueeze(0)
        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)
        x = F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))
        x = self.dropout(x)
        x = F.relu(self.gcn2(x, batch_edge_index, batch_edge_attr))
        temporal_features = x.reshape(batch_size, num_timesteps, -1).permute(1, 0, 2)
        return self.rnn(temporal_features)

# --- Component 2: LuongAttention (No changes) ---
class LuongAttention(nn.Module):
    # ... (no changes here, same as your code)
    def __init__(self, enc_dim, dec_dim):
        super().__init__()
        self.attn = nn.Linear(enc_dim, dec_dim)
    def forward(self, decoder_hidden, encoder_outputs):
        scores = torch.bmm(self.attn(encoder_outputs).permute(1, 0, 2), decoder_hidden.permute(1, 2, 0))
        attn_weights = F.softmax(scores, dim=1)
        context = torch.bmm(attn_weights.permute(0, 2, 1), encoder_outputs.permute(1, 0, 2))
        return context, attn_weights.squeeze(-1)

# --- Component 3: MetadataEncoder ---
import torch
import torch.nn as nn

class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_categories, num_objects, 
                 color_emb_dim=16, category_emb_dim=32, object_feature_dim=128):
        super().__init__()
        self.color_embedding = nn.Embedding(num_colors, color_emb_dim)
        self.category_embedding = nn.Embedding(num_categories, category_emb_dim)
        
        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, object_feature_dim)
        )
        
        self.output_dim = color_emb_dim + category_emb_dim + object_feature_dim


    def forward(self, metadata):

        # Slice the tensor into its components
        color_ids = metadata[:, 0].long()
        category_ids = metadata[:, 1].long()
        object_features_raw = metadata[:, 2:]
        
        # Ensure the input to the linear layer is a float tensor
        object_features_raw = object_features_raw.float()

        # Get embeddings for categorical features
        color_vec = self.color_embedding(color_ids)
        category_vec = self.category_embedding(category_ids)
        
        # Process the multi-hot object vector through the MLP
        object_vec = self.object_processor(object_features_raw)

        # Concatenate all features into a single vector
        combined_features = torch.cat([color_vec, category_vec, object_vec], dim=1)
        
        return combined_features

# <-- MODIFIED: Replace your entire Decoder class with this one
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, meta_features_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        enc_dim = enc_hidden * 2
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = LuongAttention(enc_dim, dec_hidden)
        # The RNN input now includes the metadata features dimension
        self.rnn = nn.GRU(emb_dim + enc_dim + meta_features_dim, dec_hidden, num_layers, dropout=dropout if num_layers > 1 else 0)
        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        # The bridge no longer needs to process metadata, only the encoder hidden state
        self.bridge = nn.Linear(enc_dim, dec_hidden)

    def init_hidden(self, encoder_hidden_cat):
        # Only uses the EEG features to initialize the hidden state
        return torch.tanh(self.bridge(encoder_hidden_cat))

    def forward(self, token, decoder_hidden, encoder_outputs, meta_features):
        token = token.unsqueeze(0)
        embedded = self.dropout(self.embedding(token))
        context, _ = self.attention(decoder_hidden[-1].unsqueeze(0), encoder_outputs)
        
        # meta_features are repeated for each token in the sequence (here, just 1)
        meta_features_unsqueezed = meta_features.unsqueeze(0)
        
        # Concatenate metadata with embedding and context at every timestep
        rnn_input = torch.cat((embedded, context.permute(1,0,2), meta_features_unsqueezed), dim=2)
        
        output, hidden = self.rnn(rnn_input, decoder_hidden)
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden


class Seq2Seq(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_categories, num_objects, enc_hidden=256, dec_hidden=256, 
                 pad_id=0, dropout=0.2, color_emb_dim=16, category_emb_dim=32, object_feature_dim=128):
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(enc_hidden=enc_hidden, dropout=dropout)
        self.meta_encoder = MetadataEncoder(num_colors, num_categories, num_objects, 
                                            color_emb_dim, category_emb_dim, object_feature_dim)
        
        meta_features_dim = self.meta_encoder.output_dim
        
        self.decoder = Decoder(text_vocab_size, 256, enc_hidden, dec_hidden, 
                               meta_features_dim, 2, pad_id, dropout)
        
        # The input to the meta_head is the encoder's output features
        enc_dim = enc_hidden * 2
        self.meta_head = nn.Sequential(
            nn.Linear(enc_dim, 256),
            nn.ReLU(),
            nn.LayerNorm(256), # Add LayerNorm for stability
            nn.Dropout(0.3),
            # The output size is reduced by 1 (removed motion score)
            nn.Linear(256, num_colors + num_categories + num_objects)
        )
        self.num_colors = num_colors
        self.num_categories = num_categories
        self.num_objects = num_objects

    def forward(self, eeg, metadata, target_text, edge_index, edge_attr, teacher_forcing_ratio=0.5):
        encoder_outputs, encoder_hidden = self.encoder(eeg, edge_index, edge_attr)
        meta_features = self.meta_encoder(metadata)
        
        forward_h = encoder_hidden[0::2]
        backward_h = encoder_hidden[1::2]
        encoder_hidden_cat = torch.cat([forward_h, backward_h], dim=2)
        
        # Initialize decoder hidden state using ONLY encoder features
        decoder_hidden = self.decoder.init_hidden(encoder_hidden_cat)
        
        batch_size = eeg.shape[0]
        target_len = target_text.shape[1]
        outputs = torch.zeros(target_len, batch_size, self.decoder.vocab_size).to(eeg.device)
        decoder_input = target_text[:, 0]
        
        for t in range(1, target_len):
            # Pass meta_features to the decoder at every timestep
            output, decoder_hidden = self.decoder(decoder_input, decoder_hidden, encoder_outputs, meta_features)
            outputs[t] = output
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)
            decoder_input = target_text[:, t] if teacher_force else top1
        
        # Predict metadata using ONLY the final encoder hidden state
        meta_preds = self.meta_head(encoder_hidden_cat[-1])
        
        # Slice the predictions for each metadata component
        pred_color = meta_preds[:, :self.num_colors]
        pred_category = meta_preds[:, self.num_colors:self.num_colors + self.num_categories]
        # The object prediction is now the rest of the tensor
        pred_object = meta_preds[:, self.num_colors + self.num_categories:]
        
        # Return without the motion prediction
        return outputs[1:].permute(1, 0, 2), pred_color, pred_category, pred_object

In [3]:
# --- Model Instantiation ---
model = Seq2Seq(
    text_vocab_size=TEXT_VOCAB_SIZE,
    num_colors=NUM_COLORS,
    num_categories=NUM_CATEGORIES,
    num_objects=NUM_OBJECTS,
    pad_id=PAD_ID,
    dropout=0.2
).to(device)

print(f"Model instantiated on '{device}'.")
print(f"Total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Model instantiated on 'cuda'.
Total parameters: 19,525,097


In [15]:
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import time
import math

# --- Training Setup ---
text_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
color_criterion = nn.CrossEntropyLoss()
category_criterion = nn.CrossEntropyLoss()
object_criterion = nn.BCEWithLogitsLoss()

optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)
scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.2, patience=2, verbose=True)


# --- Training and Evaluation Functions (Corrected) ---

def train_one_epoch(model, loader, optimizer, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr):
    model.train()
    total_loss = 0.0
    progress_bar = tqdm(loader, desc="Training", leave=False)
    
    # --- THIS IS THE FIX ---
    # The variable names now correctly match the data order from the loader: (EEG, Metadata, Text)
    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)
        
        optimizer.zero_grad()
        
        # Now this call correctly passes the right data to the right arguments
        text_logits, pred_color, pred_category, pred_object = model(eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr, teacher_forcing_ratio=0.5)
        
        text_loss = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        color_loss = color_criterion(pred_color, meta_b[:, 0].long())
        category_loss = category_criterion(pred_category, meta_b[:, 1].long())
        object_loss = object_criterion(pred_object, meta_b[:, 2:])
        
        loss = text_loss + 0.1 * (color_loss + category_loss) + 0.2 * object_loss
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        
        progress_bar.set_postfix(text_loss=text_loss.item(), meta_loss=loss.item() - text_loss.item())
        
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr):
    model.eval()
    total_loss = 0.0
    
    # --- THIS IS THE FIX ---
    # The variable names now correctly match the data order from the loader: (EEG, Metadata, Text)
    for eeg_b, meta_b, txt_b in loader:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)
        
        # Now this call correctly passes the right data to the right arguments
        text_logits, pred_color, pred_category, pred_object = model(eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr, teacher_forcing_ratio=0.0)
        
        text_loss = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        color_loss = color_criterion(pred_color, meta_b[:, 0].long())
        category_loss = category_criterion(pred_category, meta_b[:, 1].long())
        object_loss = object_criterion(pred_object, meta_b[:, 2:])
        
        loss = text_loss + 0.1 * (color_loss + category_loss) + 0.2 * object_loss
        total_loss += loss.item()
        
    return total_loss / len(loader)

# --- Training Loop ---
EPOCHS = 20
best_val_loss = float('inf')
print("\n--- Starting Training ---")
for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    
    train_loss = train_one_epoch(model, train_loader, optimizer, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr)
    val_loss = evaluate(model, val_loader, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr)
    
    scheduler.step(val_loss)
    end_time = time.time()
    formatted_time = f"{int((end_time - start_time) // 60):02d}m {int((end_time - start_time) % 60):02d}s"
    
    print(f"\nEpoch {epoch:02d}/{EPOCHS} | Time: {formatted_time}")
    print(f"\tTrain Loss: {train_loss:.4f}")
    print(f"\t Val. Loss: {val_loss:.4f} | Val. Perplexity: {math.exp(val_loss):7.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'eeg-meta-text-spatiotemporal-phase5-model.pt')
        print("\t-> Validation loss improved, saving new best model.")

print("\n--- Training Complete ---")


--- Starting Training ---


Training:   0%|          | 0/700 [00:00<?, ?it/s]


Epoch 01/20 | Time: 06m 02s
	Train Loss: 6.9567
	 Val. Loss: 5.5113 | Val. Perplexity: 247.4682
	-> Validation loss improved, saving new best model.


Training:   0%|          | 0/700 [00:00<?, ?it/s]


Epoch 02/20 | Time: 05m 54s
	Train Loss: 5.3540
	 Val. Loss: 5.2320 | Val. Perplexity: 187.1659
	-> Validation loss improved, saving new best model.


...


Epoch 18/20 | Time: 05m 56s
	Train Loss: 3.3475
	 Val. Loss: 4.6434 | Val. Perplexity: 103.9015


Training:   0%|          | 0/700 [00:00<?, ?it/s]


Epoch 19/20 | Time: 05m 56s
	Train Loss: 3.2923
	 Val. Loss: 4.6177 | Val. Perplexity: 101.2594
	-> Validation loss improved, saving new best model.


Training:   0%|          | 0/700 [00:00<?, ?it/s]


Epoch 20/20 | Time: 05m 56s
	Train Loss: 3.2153
	 Val. Loss: 4.6027 | Val. Perplexity: 99.7518
	-> Validation loss improved, saving new best model.

--- Training Complete ---


In [4]:
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer


class EEGTextMetaDataset(Dataset):
    def __init__(self, eeg_dir, metadata_dir, tokenizer, max_length=64, use_emotional_tone=True):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.use_emotional_tone = use_emotional_tone

        # -------------------------
        # 1. Load EEG files (all subjects)
        # -------------------------
        eeg_files = []
        for root, dirs, files in os.walk(eeg_dir):
            for f in files:
                if f.endswith(".npy") and "_preprocessed" in f:
                    eeg_files.append(os.path.join(root, f))
        eeg_files = sorted(eeg_files)

        if not eeg_files:
            raise FileNotFoundError(f"No EEG .npy files found in {eeg_dir}")

        self.eeg_file_paths = eeg_files
        self.eeg_data_list = []
        self.index_map = []

        for subj_idx, path in enumerate(self.eeg_file_paths):
            eeg = np.load(path, mmap_mode='r')
            assert eeg.ndim == 3 and eeg.shape[1:] == (62, 400), \
                f"EEG file {path} has shape {eeg.shape}, expected (*, 62, 400)"
            self.eeg_data_list.append(eeg)
            n_samples = eeg.shape[0]
            self.index_map.extend([(subj_idx, i) for i in range(n_samples)])

        total_samples = len(self.index_map)
        print(f"Found {len(self.eeg_file_paths)} EEG files → Total samples: {total_samples}")

        # -------------------------
        # 2. Load Metadata JSONs
        # -------------------------
        metadata_files = sorted(
            [os.path.join(dp, f)
             for dp, dn, filenames in os.walk(metadata_dir)
             for f in filenames if f.endswith(".json")]
        )
        if not metadata_files:
            raise FileNotFoundError(f"No metadata JSON files found in {metadata_dir}")

        self.metadata_list = []
        for fpath in metadata_files:
            with open(fpath, 'r', encoding='utf-8') as f:
                meta = json.load(f)

                # --- Assertion for essential content ---
                assert "semantic_features" in meta and "scene_category" in meta["semantic_features"], \
                    f"Missing scene_category in {fpath}"
                assert "visual_attributes" in meta and "major_colors" in meta["visual_attributes"], \
                    f"Missing major_colors in {fpath}"
                
                # Robustly handle missing 'objects' key
                if "objects" not in meta.get("semantic_features", {}):
                    meta["semantic_features"]["objects"] = []

                self.metadata_list.append(meta)

        base_count = len(self.metadata_list)
        print(f"Loaded {base_count} metadata JSON files")

        # -------------------------
        # 3. Build base captions
        # -------------------------
        base_captions = []
        for meta in self.metadata_list:
            caption_text = meta["caption"]["text"]
            if self.use_emotional_tone and "emotional_tone" in meta["caption"]:
                caption_text += f". Tone: {meta['caption']['emotional_tone']}"
            base_captions.append(caption_text)

        # Repeat for each subject
        num_subjects = len(self.eeg_file_paths)
        self.captions = base_captions * num_subjects
        self.metadata_repeated = self.metadata_list * num_subjects

        assert len(self.captions) == len(self.metadata_repeated) == len(self.index_map), \
            "Mismatch after repeating captions and metadata for subjects"

        # -------------------------
        # 4. Encode metadata categories (Scene, Color, and Objects)
        # -------------------------
        scene_categories = sorted(list({m["semantic_features"]["scene_category"] for m in self.metadata_list}))
        colors = sorted(list({m["visual_attributes"]["major_colors"][0]["color"].split()[0]
                               for m in self.metadata_list}))
        all_objects = set()
        for m in self.metadata_list:
            all_objects.update(m["semantic_features"]["objects"])
        objects_vocab = sorted(list(all_objects))

        self.scene_to_id = {scene: i for i, scene in enumerate(scene_categories)}
        self.color_to_id = {c: i for i, c in enumerate(colors)}
        self.object_to_id = {obj: i for i, obj in enumerate(objects_vocab)}
        
        # Create inverse mapping for easier lookup
        self.id_to_scene = {i: scene for scene, i in self.scene_to_id.items()}
        self.id_to_color = {i: c for c, i in self.color_to_id.items()}
        self.id_to_object = {i: obj for obj, i in self.object_to_id.items()}

        print(f"Scene categories: {len(self.scene_to_id)} | Colors: {len(self.color_to_id)} | Objects: {len(self.object_to_id)}")
    
    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):
        subj_idx, local_idx = self.index_map[idx]
        eeg_tensor = torch.tensor(self.eeg_data_list[subj_idx][local_idx], dtype=torch.float32)
        caption = self.captions[idx]
        
        tokenized = self.tokenizer(
            caption, padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt"
        )
        
        # This is the line from your CORRECT __getitem__ method
        input_ids = tokenized['input_ids'].squeeze(0)
        
        meta = self.metadata_repeated[idx]
        
        scene_id = self.scene_to_id[meta["semantic_features"]["scene_category"]]
        color_id = self.color_to_id[meta["visual_attributes"]["major_colors"][0]["color"].split()[0]]
        scalar_meta_tensor = torch.tensor([scene_id, color_id], dtype=torch.float32)

        objects_present = meta["semantic_features"]["objects"]
        object_multi_hot = torch.zeros(len(self.object_to_id), dtype=torch.float32)
        for obj in objects_present:
            if obj in self.object_to_id:
                obj_id = self.object_to_id[obj]
                object_multi_hot[obj_id] = 1.0
        
        metadata_tensor = torch.cat((scalar_meta_tensor, object_multi_hot))
        
        # This is the CORRECT return statement
        return eeg_tensor, input_ids, metadata_tensor

In [5]:
import torch
import torch.nn.functional as F
import random
import json # <--- NEW: Import json library

# --- Inference ---
# Assuming the model and data loaders have been set up in previous cells.

# Load the trained model weights
checkpoint_path = 'eeg-meta-text-spatiotemporal-phase5-model.pt'
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
print("Best spatiotemporal model loaded successfully.")

# Define the inference function
@torch.no_grad()
def generate_with_metadata(model, eeg_signal, meta_signal, edge_index, edge_attr, beam_width=5, max_len=100):
    model.eval()
    eeg_signal = eeg_signal.unsqueeze(0).to(device)
    meta_signal = meta_signal.unsqueeze(0).to(device)
    
    # 1. Get Encoder outputs and Metadata features
    encoder_outputs, encoder_hidden = model.encoder(eeg_signal, edge_index, edge_attr)
    meta_features = model.meta_encoder(meta_signal)
    
    # 2. Reshape encoder hidden state
    num_layers = model.decoder.rnn.num_layers
    forward_h = encoder_hidden[0::2]
    backward_h = encoder_hidden[1::2]
    encoder_hidden_cat = torch.cat([forward_h, backward_h], dim=2)
    
    # 3. MODIFIED: Initialize decoder hidden state from EEG features ONLY
    decoder_hidden = model.decoder.init_hidden(encoder_hidden_cat)
    
    # 4. Predict metadata from the EEG features
    meta_preds = model.meta_head(encoder_hidden_cat[-1]) # Use last layer's hidden state
    pred_color = meta_preds[0, :model.num_colors].argmax().item()
    pred_category = meta_preds[0, model.num_colors:model.num_colors + model.num_categories].argmax().item()
    pred_object = meta_preds[0, model.num_colors + model.num_categories:].argmax().item() # Corrected slicing
    
    # --- Beam Search Loop ---
    beams = [([SOS_ID], 0.0, decoder_hidden)]
    
    for _ in range(max_len):
        new_beams = []
        for seq, score, hidden in beams:
            if seq[-1] == EOS_ID:
                new_beams.append((seq, score, hidden))
                continue
                
            input_token = torch.tensor([seq[-1]], device=device)
            
            # 5. MODIFIED: Pass meta_features into the decoder's forward call
            prediction, new_hidden = model.decoder(input_token, hidden, encoder_outputs, meta_features)
            
            log_probs = F.log_softmax(prediction, dim=-1).squeeze()
            top_log_probs, top_ids = torch.topk(log_probs, beam_width)
            
            for i in range(beam_width):
                new_seq = seq + [top_ids[i].item()]
                new_score = score + top_log_probs[i].item()
                new_beams.append((new_seq, new_score, new_hidden))

        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
        
        # Stop if the top beam has ended
        if beams[0][0][-1] == EOS_ID:
            break
            
    predicted_text_ids = beams[0][0][1:-1] # Exclude SOS and EOS tokens
    predicted_text = tokenizer.decode(predicted_text_ids, skip_special_tokens=True)

    return predicted_text, pred_color, pred_category, pred_object


# <--- NEW: Load the object mapping from JSON ---
OBJECT_MAPPING_FILE = "/home/poorna/data/object_id_to_name.json"
try:
    with open(OBJECT_MAPPING_FILE, 'r') as f:
        object_mapping = json.load(f)
    print(f"Object mapping '{OBJECT_MAPPING_FILE}' loaded successfully.")
except FileNotFoundError:
    print(f"Warning: '{OBJECT_MAPPING_FILE}' not found. Object names will not be displayed.")
    object_mapping = {} # Use an empty dict as a fallback

# --- ADD THIS DEBUG BLOCK BEFORE YOUR INFERENCE LOOP ---
print("--- Running Debug Sanity Check ---")
try:
    # Get just the first sample from the dataset
    eeg_check, ids_check, meta_check = test_ds[0]
    
    # Print the shapes and types to be absolutely sure
    print(f"Item 1 from dataset (eeg_sample):   Shape={eeg_check.shape}, Dtype={eeg_check.dtype}")
    print(f"Item 2 from dataset (true_text_ids): Shape={ids_check.shape}, Dtype={ids_check.dtype}")
    print(f"Item 3 from dataset (meta_sample):   Shape={meta_check.shape}, Dtype={meta_check.dtype}")

except Exception as e:
    print(f"Error during sanity check: {e}")
    print("This might happen if the number of returned items is not 3.")
print("------------------------------------")
# --- END OF DEBUG BLOCK ---

# --- Run Inference and Print Results ---
NUM_SAMPLES = 20 
print(f"\n--- Running Inference on the First {NUM_SAMPLES} Samples ---")

for i in range(NUM_SAMPLES):
    eeg_sample, meta_sample, true_text_ids = test_ds[i] # Swapped order to match your Dataset return
    
    # --- THIS IS THE CORRECTED SECTION ---

    # Extract true metadata correctly from the combined tensor
    true_category_id = int(meta_sample[0].item()) # Index 0 is scene/category
    true_color_id = int(meta_sample[1].item())    # Index 1 is color
    
    # Extract the multi-hot vector for objects
    true_object_vector = meta_sample[2:]
    
    # Find all indices that are '1' (i.e., all objects that are present)
    true_object_ids_tensors = true_object_vector.nonzero(as_tuple=True)[0]
    
    # Convert tensor indices to a list of names for printing
    true_object_names = [object_mapping.get(str(id.item()), f"Unknown ID: {id.item()}") for id in true_object_ids_tensors]
    if not true_object_names:
        true_object_names = ["None"] # Handle cases where no objects are tagged

    # --- END OF CORRECTIONS ---

    predicted_text, pred_color, pred_category, pred_object = generate_with_metadata(
        model, eeg_sample, meta_sample, granger_edge_index, granger_edge_attr
    )
    
    true_text = tokenizer.decode(true_text_ids, skip_special_tokens=True)
    
    # Use .get() for safe lookup, providing a fallback if an ID isn't found
    pred_object_name = object_mapping.get(str(pred_object), f"Unknown ID: {pred_object}")

    print(f"\n--- Sample {i+1}/{NUM_SAMPLES} (Index: {i}) ---")
    print(f"GROUND TRUTH TEXT: {true_text}")
    print(f"MODEL PREDICTION TEXT: {predicted_text}")
    print("\nMETADATA PREDICTION:")
    print(f"  Color ID:      Truth={true_color_id}, Predicted={pred_color}")
    print(f"  Category ID:   Truth={true_category_id}, Predicted={pred_category}")
    # MODIFIED: Updated print statement to show a list of true objects
    print(f"  Object(s):     Truth={', '.join(true_object_names)}, Predicted='{pred_object_name}' ({pred_object})")

Best spatiotemporal model loaded successfully.
Object mapping '/home/poorna/data/object_id_to_name.json' loaded successfully.
--- Running Debug Sanity Check ---
Item 1 from dataset (eeg_sample):   Shape=torch.Size([62, 400]), Dtype=torch.float32
Item 2 from dataset (true_text_ids): Shape=torch.Size([63]), Dtype=torch.float32
Item 3 from dataset (meta_sample):   Shape=torch.Size([64]), Dtype=torch.int64
------------------------------------

--- Running Inference on the First 20 Samples ---

--- Sample 1/20 (Index: 0) ---
GROUND TRUTH TEXT: a school of orange fish swims around a vibrant coral reef.. tone : serene
MODEL PREDICTION TEXT: a sea turtle swims gracefully in a coral reef.. tone : serene

METADATA PREDICTION:
  Color ID:      Truth=36, Predicted=36
  Category ID:   Truth=67, Predicted=50
  Object(s):     Truth=fish, water, Predicted='person' (40)

--- Sample 2/20 (Index: 1) ---
GROUND TRUTH TEXT: powerful waterfalls cascade down rocky cliffs into a misty pool.. tone : awe - insp

In [7]:
# with scores

In [5]:
import torch
import torch.nn.functional as F
import random
import json
import evaluate # <--- NEW: Import evaluate library

# --- Load Model and Mappings (Your existing code) ---
# Assuming 'model', 'test_ds', 'tokenizer', 'device', 'granger_edge_index', 
# 'granger_edge_attr', 'object_mapping', 'SOS_ID', 'EOS_ID' are already defined.

checkpoint_path = 'eeg-meta-text-spatiotemporal-phase5-model.pt'
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
print("Best spatiotemporal model loaded successfully.")

OBJECT_MAPPING_FILE = "/home/poorna/data/object_id_to_name.json"
try:
    with open(OBJECT_MAPPING_FILE, 'r') as f:
        object_mapping = json.load(f)
    print(f"Object mapping '{OBJECT_MAPPING_FILE}' loaded successfully.")
except FileNotFoundError:
    print(f"Warning: '{OBJECT_MAPPING_FILE}' not found. Object names will not be displayed.")
    object_mapping = {}

# --- Inference Function (Your existing, corrected function) ---
@torch.no_grad()
def generate_with_metadata(model, eeg_signal, meta_signal, edge_index, edge_attr, beam_width=5, max_len=100):
    model.eval()
    eeg_signal = eeg_signal.unsqueeze(0).to(device)
    meta_signal = meta_signal.unsqueeze(0).to(device)
    
    encoder_outputs, encoder_hidden = model.encoder(eeg_signal, edge_index, edge_attr)
    meta_features = model.meta_encoder(meta_signal)
    
    num_layers = model.decoder.rnn.num_layers
    forward_h = encoder_hidden[0::2]
    backward_h = encoder_hidden[1::2]
    encoder_hidden_cat = torch.cat([forward_h, backward_h], dim=2)
    
    decoder_hidden = model.decoder.init_hidden(encoder_hidden_cat)
    
    meta_preds = model.meta_head(encoder_hidden_cat[-1])
    pred_color = meta_preds[0, :model.num_colors].argmax().item()
    pred_category = meta_preds[0, model.num_colors:model.num_colors + model.num_categories].argmax().item()
    pred_object = meta_preds[0, model.num_colors + model.num_categories:].argmax().item()
    
    beams = [([SOS_ID], 0.0, decoder_hidden)]
    
    for _ in range(max_len):
        new_beams = []
        for seq, score, hidden in beams:
            if seq[-1] == EOS_ID:
                new_beams.append((seq, score, hidden))
                continue
            
            input_token = torch.tensor([seq[-1]], device=device)
            prediction, new_hidden = model.decoder(input_token, hidden, encoder_outputs, meta_features)
            
            log_probs = F.log_softmax(prediction, dim=-1).squeeze()
            top_log_probs, top_ids = torch.topk(log_probs, beam_width)
            
            for i in range(beam_width):
                new_seq = seq + [top_ids[i].item()]
                new_score = score + top_log_probs[i].item()
                new_beams.append((new_seq, new_score, new_hidden))

        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
        
        if beams[0][0][-1] == EOS_ID:
            break
            
    predicted_text_ids = beams[0][0][1:-1]
    predicted_text = tokenizer.decode(predicted_text_ids, skip_special_tokens=True)

    return predicted_text, pred_color, pred_category, pred_object

# --- Run Inference, Print Results, and Calculate Scores ---

NUM_SAMPLES = 20
print(f"\n--- Running Inference on the First {NUM_SAMPLES} Samples ---")

# <--- NEW: Create lists to store predictions and references ---
predictions = []
references = []

for i in range(NUM_SAMPLES):
    eeg_sample, meta_sample, true_text_ids = test_ds[i]
    
    true_category_id = int(meta_sample[0].item())
    true_color_id = int(meta_sample[1].item())
    true_object_vector = meta_sample[2:]
    true_object_ids_tensors = true_object_vector.nonzero(as_tuple=True)[0]
    true_object_names = [object_mapping.get(str(id.item()), f"Unknown ID: {id.item()}") for id in true_object_ids_tensors]
    if not true_object_names:
        true_object_names = ["None"]

    predicted_text, pred_color, pred_category, pred_object = generate_with_metadata(
        model, eeg_sample, meta_sample, granger_edge_index, granger_edge_attr
    )
    
    true_text = tokenizer.decode(true_text_ids, skip_special_tokens=True)
    
    # <--- NEW: Add the results to our lists ---
    predictions.append(predicted_text)
    references.append(true_text) # For ROUGE
    
    pred_object_name = object_mapping.get(str(pred_object), f"Unknown ID: {pred_object}")

    print(f"\n--- Sample {i+1}/{NUM_SAMPLES} (Index: {i}) ---")
    print(f"GROUND TRUTH TEXT: {true_text}")
    print(f"MODEL PREDICTION TEXT: {predicted_text}")
    print("\nMETADATA PREDICTION:")
    print(f"  Color ID:      Truth={true_color_id}, Predicted={pred_color}")
    print(f"  Category ID:   Truth={true_category_id}, Predicted={pred_category}")
    print(f"  Object(s):     Truth={', '.join(true_object_names)}, Predicted='{pred_object_name}' ({pred_object})")

# <--- NEW: Calculate and print the scores after the loop ---
print("\n--- Evaluation Metrics ---")

# Calculate BLEU score
bleu_metric = evaluate.load('bleu')
# Note: BLEU expects references to be a list of lists.
bleu_results = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])
print(f"BLEU Score: {bleu_results['bleu']:.4f}")

# Calculate ROUGE scores
rouge_metric = evaluate.load('rouge')
rouge_results = rouge_metric.compute(predictions=predictions, references=references)
print(f"ROUGE-1 Score: {rouge_results['rouge1']:.4f}")
print(f"ROUGE-2 Score: {rouge_results['rouge2']:.4f}")
print(f"ROUGE-L Score: {rouge_results['rougeL']:.4f}")

Best spatiotemporal model loaded successfully.
Object mapping '/home/poorna/data/object_id_to_name.json' loaded successfully.

--- Running Inference on the First 20 Samples ---

--- Sample 1/20 (Index: 0) ---
GROUND TRUTH TEXT: a school of orange fish swims around a vibrant coral reef.. tone : serene
MODEL PREDICTION TEXT: a sea turtle swims gracefully in a coral reef.. tone : serene

METADATA PREDICTION:
  Color ID:      Truth=36, Predicted=36
  Category ID:   Truth=67, Predicted=50
  Object(s):     Truth=fish, water, Predicted='person' (40)

--- Sample 2/20 (Index: 1) ---
GROUND TRUTH TEXT: powerful waterfalls cascade down rocky cliffs into a misty pool.. tone : awe - inspiring
MODEL PREDICTION TEXT: water rushes over rocks in a sandy beach.. tone : serene

METADATA PREDICTION:
  Color ID:      Truth=50, Predicted=36
  Category ID:   Truth=36, Predicted=6
  Object(s):     Truth=drink, mountain, rock, tree, water, Predicted='person' (40)

--- Sample 3/20 (Index: 2) ---
GROUND TRUTH TE

BLEU Score: 0.3514


ROUGE-1 Score: 0.4588
ROUGE-2 Score: 0.2587
ROUGE-L Score: 0.4524
